#Validação cruzada e tunning de hiperparametro
---
**Aula Prática 19**: Validação cruzada e tunning de hiperparâmetro


**Objetivo**: Identificar o melhor conjunto de hiperparâmetros com validação cruzada.


Banco de dados:


**Breast cancer wisconsin dataset**


Disponível via sklearn


> Features are computed from a digitized image of a fine needle aspirate (FNA) of a breast mass.  They describe characteristics of the cell nuclei present in the image.
>
> 1) ID number
>
> 2) Diagnosis (0 = malignant, 1 = benign)
>
> 3-32)
>
> Ten real-valued features are computed for each cell nucleus:
>
> a) radius (mean of distances from center to points on the perimeter)
>
> b) texture (standard deviation of gray-scale values)
>
> c) perimeter
>
> d) area
>
> e) smoothness (local variation in radius lengths)
>
> f) compactness (perimeter^2 / area - 1.0)
>
> g) concavity (severity of concave portions of the contour)
>
> h) concave points (number of concave portions of the contour)
>
> i) symmetry
>
> j) fractal dimension ("coastline approximation" - 1)

##Import das principais funções e leitura dos dados


---



In [9]:
import pandas as pd
import numpy as np
from sklearn import datasets

In [2]:
data = datasets.load_breast_cancer()

In [3]:
df = pd.DataFrame(data.data, columns=data.feature_names)

In [4]:
target = pd.DataFrame(data.target, columns=['Target'])
df = pd.concat([df, target], axis=1)

In [5]:
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,Target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [6]:
df.shape

(569, 31)

In [7]:
df.dtypes

mean radius                float64
mean texture               float64
mean perimeter             float64
mean area                  float64
mean smoothness            float64
mean compactness           float64
mean concavity             float64
mean concave points        float64
mean symmetry              float64
mean fractal dimension     float64
radius error               float64
texture error              float64
perimeter error            float64
area error                 float64
smoothness error           float64
compactness error          float64
concavity error            float64
concave points error       float64
symmetry error             float64
fractal dimension error    float64
worst radius               float64
worst texture              float64
worst perimeter            float64
worst area                 float64
worst smoothness           float64
worst compactness          float64
worst concavity            float64
worst concave points       float64
worst symmetry      

In [8]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
mean radius,569.0,14.127292,3.524049,6.981000,11.700000,13.370000,15.780000,28.11000
mean texture,569.0,19.289649,4.301036,9.710000,16.170000,18.840000,21.800000,39.28000
mean perimeter,569.0,91.969033,24.298981,43.790000,75.170000,86.240000,104.100000,188.50000
mean area,569.0,654.889104,351.914129,143.500000,420.300000,551.100000,782.700000,2501.00000
mean smoothness,569.0,0.096360,0.014064,0.052630,0.086370,0.095870,0.105300,0.16340
mean compactness,569.0,0.104341,0.052813,0.019380,0.064920,0.092630,0.130400,0.34540
mean concavity,569.0,0.088799,0.079720,0.000000,0.029560,0.061540,0.130700,0.42680
mean concave points,569.0,0.048919,0.038803,0.000000,0.020310,0.033500,0.074000,0.20120
mean symmetry,569.0,0.181162,0.027414,0.106000,0.161900,0.179200,0.195700,0.30400
mean fractal dimension,569.0,0.062798,0.007060,0.049960,0.057700,0.061540,0.066120,0.09744


### Separação do banco entre treino e teste
O primeiro passo para se treinar um modelo é separar o banco entre treino e teste. Para isso utilizaremos a função train_test_split




``` python
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=.3, random_state=15)
```
No exemplo acima X é um dataframe contendo as features do modelo e Y um dataframe com a variável target.




O parâmetro test_size controla o percentual de dados que será utilizado para teste.




O parâmetro random_state controla a aleatoriedade da geração do dado, permitindo que ao executar o código seja gerado os mesmos bancos de treino e teste.




É importante separar o banco entre treino e teste, pois utilizaremos o banco de treino para treinar modelos e o banco de teste para avaliar os modelos.


### Validação cruzada
Para realizar a validação cruzada separamos o banco de treino em pequenos grupos chamados folds, cada um será utilizado como um banco de treino para avaliarmos a consistência do modelo.


``` python
from sklearn.model_selection import cross_validate
scoring = ['precision_macro', 'recall_macro']
scores = cross_validate(model, X_train, Y_train, scoring=scoring)
```


### Tuning de hiperparâmetro
Para achar o hiperparâmetro que dê o melhor resultado devemos testar diferentes combinações de hiperparâmetro para achar o melhor. Há três algoritmos para fazer:


* GridSearch: Busca por todos os possíveis valores passados pelo usuário.
* RandomSerch: Busca aleatória em um intervalo de valores determinado
* BayesianOptimization: Busca através de um otimizador.


No código a seguir iremos utilizar o GridSearch com validação cruzada


``` python
from sklearn.model_selection import GridSearchCV
parameters = {'parâmetro 1' : [0.1, 0.03, 0.01],
              'parâmetro 2': [100, 500, 50]}


clf = GridSearchCV(model, parameters)
clf.fit(X_train, Y_train)
clf.best_params_
pd.DataFrame(clf.cv_results_)
```

No código a seguir iremos utilizar o RandomSearch com validação cruzada


``` python
from sklearn.model_selection import RandomizedSearchCV
parameters = {'parâmetro 1' : scipy.stats.expon(scale=.1),
              'parâmetro 2': scipy.stats.uniform(50, 500)}


clf = RandomizedSearchCV(model, parameters)
clf.fit(X_train, Y_train)
clf.best_params_
pd.DataFrame(clf.cv_results_)
```

In [11]:
X = pd.DataFrame(data.data, columns=data.feature_names)
Y = data.target

### Exercício:


* Separe o banco entre treino e teste. Use 30% do banco para teste. Faça a quebra com todas as variáveis.
* Utilizando um classificador xgboost faça:
* Faça a validação cruzada do modelo, qual o valor médio de precisão macro
* Faça uma busca, utilizando grid search, nos parâmetros learning_rate com valores de 0.1, 0.05 e 0.01 e n_estimators com valores de 100, 500, 50. Qual é a melhor configuração?
* Aplique essa configuração no modelo e avalie os resultados.

#### Solução

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=15
)

In [14]:
from xgboost import XGBClassifier

In [15]:
model_xgb = XGBClassifier(objective='binary:logistic')

In [18]:
from sklearn.model_selection import cross_validate

scoring = ["precision_macro", "recall_macro"]
scores = cross_validate(model_xgb, X_train, Y_train, scoring=scoring)

In [19]:
scores

{'fit_time': array([0.27284384, 0.14691377, 0.12392879, 0.11593509, 0.10794044]),
 'score_time': array([0.01099324, 0.01499438, 0.01099348, 0.01299   , 0.01199102]),
 'test_precision_macro': array([0.98076923, 0.9532967 , 0.9532967 , 0.98333333, 0.90292208]),
 'test_recall_macro': array([0.96666667, 0.94      , 0.94      , 0.99      , 0.92210884])}

In [22]:
scores['test_precision_macro'].mean()

0.9547236097236098

In [24]:
from sklearn.model_selection import GridSearchCV

parameters = {"learning_rate": [0.1, 0.03, 0.01], "n_estimators": [100, 500, 50]}

clf = GridSearchCV(model_xgb, parameters)
clf.fit(X_train, Y_train)

GridSearchCV(estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=None, max_bin=None,
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                   

### Exercício:


* Utilizando um classificador xgboost faça:
* Faça uma busca, utilizando random search, nos parâmetros learning_rate com distribuição exponencial com parâmetro 0.01 e n_estimators com distribuição uniforme de 50 a 500 (utilizar a distribuição randint). Qual é a melhor configuração? Faça o número de tentativas igual a 9
* Aplique essa configuração no modelo e avalie os resultados.

#### Solução

In [26]:
!pip install optuna

In [27]:
import optuna
from statistics import mean


def objective(trial):
    xg_lr = trial.suggest_float("xg_lr", 0.01, 0.1, log=True)
    xg_est = trial.suggest_int("xg_est", 50, 500, log=False)

    classifier = XGBClassifier(
        objective="binary:logistic", learning_rate=xg_lr, n_estimators=xg_est
    )
    scores = cross_validate(classifier, X_train, Y_train, scoring="accuracy")
    return mean(scores["test_score"])


In [28]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

[I 2024-05-07 21:02:00,874] A new study created in memory with name: no-name-b5bc1c14-587e-485c-afbe-8007bb276bed
[I 2024-05-07 21:02:01,934] Trial 0 finished with value: 0.9421835443037975 and parameters: {'xg_lr': 0.050632022452201816, 'xg_est': 66}. Best is trial 0 with value: 0.9421835443037975.
[I 2024-05-07 21:02:05,605] Trial 1 finished with value: 0.9422151898734177 and parameters: {'xg_lr': 0.01178216013069391, 'xg_est': 250}. Best is trial 1 with value: 0.9422151898734177.
[I 2024-05-07 21:02:10,877] Trial 2 finished with value: 0.952246835443038 and parameters: {'xg_lr': 0.017944781292539246, 'xg_est': 438}. Best is trial 2 with value: 0.952246835443038.
[I 2024-05-07 21:02:16,384] Trial 3 finished with value: 0.949746835443038 and parameters: {'xg_lr': 0.010905260156084465, 'xg_est': 433}. Best is trial 2 with value: 0.952246835443038.
[I 2024-05-07 21:02:18,408] Trial 4 finished with value: 0.949746835443038 and parameters: {'xg_lr': 0.08765834832005569, 'xg_est': 98}. Bes

* Utilizando um classificador RandomForest faça:
* Faça uma busca, utilizando random search, nos parâmetros max_depth com distribuição uniforme discreta de 2 a 10 e n_estimators com distribuição uniforme de 50 a 500. Qual é a melhor configuração? Faça o número de tentativas igual a 9
* Aplique essa configuração no modelo e avalie os resultados.

#### Solução

Optuna

Para utilizar a otimização bayesiana iremos usar o pacote Optuna.

Para a utilização desse pacote devemos considerar os seguinte passos:
1. Definir uma função objetivo
2. Criar sugestão dos valores do hiperparâmetro utilizando o módulo trial
3. Criar um objeto de estudo

``` python
import optuna

# 1. Define an objective function to be maximized.
def objective(trial):

    # 2. Suggest values for the hyperparameters using a trial object.
    rf_max_depth = trial.suggest_int('rf_max_depth', 2, 32, log=True)
    classifier_obj = sklearn.ensemble.RandomForestClassifier(max_depth=rf_max_depth, n_estimators=10)
    ...
    return accuracy

# 3. Create a study object and optimize the objective function.
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)
```



* Utilizando um classificador xgboost faça:
* Faça uma busca, utilizando optuna, nos parâmetros learning_rate com distribuição log uniform de 0.01 a 0.1 e n_estimators com distribuição uniforme de 50 a 500. Qual é a melhor configuração? Faça o número de tentativas igual a 9
* Aplique essa configuração no modelo e avalie os resultados.

[Link com documentação dos trial](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html#optuna.trial.Trial.suggest_float)

#### Solução